# Chapter 06 · Artificial Intelligence & Machine Learning

Week 07 · Python Intermediate · Google Colab  
각 Chapter는 **완전한 독립 실행 코드**로 구성되어 있습니다. 위에서 아래로 실행하고, 각 코드를 변경해 예측 결과를 확인하세요.

## 준비 · 라이브러리 확인

Colab에 기본 설치되어 있지 않은 경우에만 설치합니다. GPU는 필요하지 않습니다.

In [ ]:
%pip install -q numpy matplotlib scikit-learn

import sklearn, numpy, matplotlib
print("scikit-learn:", sklearn.__version__)

## 6.1 · Introduction to Artificial Intelligence

키워드 규칙을 사람이 직접 작성하여 메시지를 분류합니다. **머신러닝 모델을 학습하는 코드는 아닙니다.**

코드 실행 후 출력 및 그래프를 확인하세요.

In [ ]:
"""6.1 | 규칙 기반 메시지 분류. 머신러닝 학습을 수행하지 않습니다."""

def classify_message(message):
    keywords = ["무료", "당첨", "광고"]
    for word in keywords:
        if word in message:
            return "광고 의심"
    return "일반 메시지"

messages = [
    "오늘 수업 자료를 공유합니다.",
    "무료 쿠폰에 당첨되셨습니다.",
    "무료 공개 강의 안내입니다.",
]

for message in messages:
    print(message, "→", classify_message(message))

print("\n확인 활동: keywords에 '할인'을 추가하고 메시지를 다시 분류해 보세요.")


## 6.2 · How Machine Learning Works

데이터 준비 → Train/Test 분리 → 모델 생성 → `fit()` → `predict()` → MAE 평가 → 새 입력 예측

코드 실행 후 출력 및 그래프를 확인하세요.
**모델링 기본 절차**: ① `from ... import ...`로 클래스 가져오기 → ② `X`, `y` 준비 → ③ Train/Test 분리 → ④ `model = LinearRegression()`으로 객체 생성 → ⑤ `fit()` 학습 → ⑥ `predict()` 예측 → ⑦ 정답과 비교하여 MAE 평가. 각 단계는 별개의 작업이며, 모델 생성만으로 학습이 완료되지는 않습니다.


In [ ]:
"""6.2 | 공부 시간 → 점수 예측. 모델 불러오기·준비·분할·생성·학습·평가·예측."""
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# 1. 데이터 준비 — 각 행은 학생 한 명, 입력 특징은 공부 시간 한 개
X = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]]
y = [42, 48, 53, 57, 62, 70, 74, 77, 84, 89]

# 2. 학습용 80% / 테스트용 20% 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"데이터: Train {len(X_train)}개 / Test {len(X_test)}개")

# 3. 모델 생성: 아직 학습하지 않은 LinearRegression 객체
model = LinearRegression()

# 4. 학습: 학습용 입력과 정답만 전달
model.fit(X_train, y_train)

# 5. 테스트 데이터 예측 및 평가
pred_test = model.predict(X_test)
print("\n[테스트 데이터: 실제 점수 vs 예측 점수]")
for hours, actual, predicted in zip(X_test, y_test, pred_test):
    print(f"{hours[0]}시간: 실제 {actual}점 / 예측 {predicted:.1f}점")
print(f"MAE: {mean_absolute_error(y_test, pred_test):.2f}점")

# 6. 새로운 입력으로 예측 — 이 값의 정답은 알려져 있지 않음
hours = 6.5
prediction = model.predict([[hours]])[0]
print(f"\n새로운 입력 {hours}시간의 예상 점수: {prediction:.1f}점")
print(f"학습된 기울기: {model.coef_[0]:.3f}, 절편: {model.intercept_:.3f}")

# 시각화: 실제 데이터와 모델의 예측 직선
x_plot = np.linspace(1, 10, 100).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(9.6, 5.6), dpi=160)
ax.scatter(np.asarray(X_train).ravel(), y_train, s=78, label="Train (8)", zorder=3)
ax.scatter(np.asarray(X_test).ravel(), y_test, s=94, marker="s", label="Test (2)", zorder=4)
ax.plot(x_plot.ravel(), model.predict(x_plot), linewidth=2.5, label="Linear Regression")
ax.scatter([hours], [prediction], s=150, marker="*", label="New input: 6.5 h", zorder=5)
ax.set(xlabel="Study hours (h)", ylabel="Exam score (points)", title="6.2  |  Linear Regression: Train / Test / New Input")
ax.legend(loc="upper left")
ax.grid(alpha=0.2)
fig.tight_layout()
output = Path("assets")
output.mkdir(exist_ok=True)
fig.savefig(output / "6_2_linear_regression.png", bbox_inches="tight", facecolor="white")
plt.show()


## 6.3 · Classification & Prediction

붓꽃 데이터에서 꽃받침 특징 두 개를 사용하고, KNN 분류기를 학습한 뒤 정확도를 확인합니다.

코드 실행 후 출력 및 그래프를 확인하세요.
회귀는 숫자를 예측하고 분류는 범주를 예측합니다. 의사결정나무와 KNN은 서로 다른 분류 알고리즘이지만 `fit()`·`predict()` 사용법은 비슷합니다. Iris 모델은 학습한 세 품종 중 하나만 출력하므로, 낯선 꽃에서도 분류 이름이 출력된다는 점에 유의하세요.


In [ ]:
"""6.3 | KNN 붓꽃 분류. 길이·너비 특징, Train/Test, 새 꽃 예측, 결정 영역."""
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1. 데이터 준비: 꽃받침 길이·너비(cm)만 사용
iris = load_iris()
X = iris.data[:, :2]
y = iris.target

# 2. 데이터 분리: 품종 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 모델 생성 (가까운 꽃 다섯 개 참조)
model = KNeighborsClassifier(n_neighbors=5)

# 4. 학습 (KNN은 학습 사례를 저장하고 거리를 예측 시점에 활용)
model.fit(X_train, y_train)

# 5. 테스트 데이터 예측 및 정확도 평가
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Train {len(X_train)}개 / Test {len(X_test)}개")
print(f"Test Accuracy: {accuracy:.3f} ({(predictions == y_test).sum()}/{len(y_test)})")

# 6. 새로운 꽃 예측
new_flower = [[5.7, 2.8]]  # 꽃받침 길이, 너비
species = iris.target_names[model.predict(new_flower)[0]]
print(f"꽃받침 {new_flower[0]} cm → {species}")

# 시각화: 모델이 구분하는 2차원 결정 영역
x_min, x_max = X[:, 0].min() - 0.35, X[:, 0].max() + 0.35
y_min, y_max = X[:, 1].min() - 0.35, X[:, 1].max() + 0.35
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 230), np.linspace(y_min, y_max, 180))
region = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(9.4, 5.8), dpi=160)
ax.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5, 2.5], cmap=ListedColormap(["#d7e9f5", "#e8dff8", "#f8e7cb"]), alpha=0.9)
for i, name in enumerate(iris.target_names):
    mask = y_train == i
    ax.scatter(X_train[mask, 0], X_train[mask, 1], s=35, label=f"Train: {name}")
ax.scatter(X_test[:, 0], X_test[:, 1], facecolors="none", edgecolors="black", linewidths=1.2, s=75, label="Test (outlined)")
ax.scatter(new_flower[0][0], new_flower[0][1], marker="*", s=185, c="black", label="New flower")
ax.set(xlabel="Sepal length (cm)", ylabel="Sepal width (cm)", title="6.3  |  KNN (k=5): Iris Classification")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.15)
fig.tight_layout()
out = Path("assets")
out.mkdir(exist_ok=True)
fig.savefig(out / "6_3_iris_knn.png", bbox_inches="tight", facecolor="white")
plt.show()


## 6.4 · Model Evaluation & Applications

Train/Validation/Test를 나누고, 검증 데이터로 특징을 선택한 뒤 최종 테스트를 한 번 수행합니다.

코드 실행 후 출력 및 그래프를 확인하세요.
설정값 `n_neighbors`는 **하이퍼파라미터**입니다. Train 데이터로 후보를 학습하고 Validation으로 후보를 선택한 뒤, Test는 마지막 성능 평가에 사용합니다. Test 결과를 보고 여러 번 모델을 고르면 성능이 낙관적으로 보일 수 있습니다.


In [ ]:
"""6.4 | Train/Validation/Test로 특징 비교 → 선택 후 Test 한 번 평가."""
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1. 정답과 전체 특징을 읽음. 아직 특징 선택에 테스트 결과를 사용하지 않음.
iris = load_iris()
X = iris.data  # 꽃받침 길이·너비 / 꽃잎 길이·너비 (4개)
y = iris.target

# 2. 전체의 20%를 마지막 테스트용으로 보관
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# 3. 남은 80%를 Train 75% / Validation 25%로 나눔
#    전체 기준 60% / 20% / 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev
)
print(f"Train: {len(X_train)} / Validation: {len(X_val)} / Test: {len(X_test)}")

# 4. 모델은 고정하고, 입력 Feature만 변경해 검증 성능을 비교
feature_sets = {
    "Sepal": [0, 1],
    "Petal": [2, 3],
}
validation_scores = {}
for name, cols in feature_sets.items():
    candidate = KNeighborsClassifier(n_neighbors=5)
    candidate.fit(X_train[:, cols], y_train)
    val_predictions = candidate.predict(X_val[:, cols])
    validation_scores[name] = accuracy_score(y_val, val_predictions)
    print(f"{name} Validation Accuracy: {validation_scores[name]:.3f}")

# 5. 검증 정확도가 높은 특징 선택. Test 데이터는 선택에 사용하지 않음.
best_name = max(validation_scores, key=validation_scores.get)
best_cols = feature_sets[best_name]
print("Selected features:", best_name)

# 6. 선택을 마친 뒤 개발용 전체(Train+Validation)로 최종 모델 학습
final_model = KNeighborsClassifier(n_neighbors=5)
final_model.fit(X_dev[:, best_cols], y_dev)

# 7. 보관했던 Test로 최종 평가
final_predictions = final_model.predict(X_test[:, best_cols])
final_acc = accuracy_score(y_test, final_predictions)
cm = confusion_matrix(y_test, final_predictions)
print(f"Final Test Accuracy: {final_acc:.3f} ({(final_predictions == y_test).sum()}/{len(y_test)})")
print("Confusion matrix (rows=actual, columns=predicted):\n", cm)

out = Path("assets")
out.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(7.9, 4.9), dpi=170)
labels = list(validation_scores)
values = [validation_scores[l] * 100 for l in labels]
bars = ax.bar(labels, values, width=0.48)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1.1, f"{v:.1f}%", ha="center", fontsize=12)
ax.set_ylim(0, 108)
ax.set_ylabel("Validation accuracy (%)")
ax.set_title("6.4  |  Feature Selection with Validation Set")
ax.grid(axis="y", alpha=0.17)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(out / "6_4_feature_comparison.png", bbox_inches="tight", facecolor="white")
plt.show()

fig, ax = plt.subplots(figsize=(7.1, 5.7), dpi=170)
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set_title(f"6.4  |  Final Test: {final_acc:.1%} accuracy")
fig.tight_layout()
fig.savefig(out / "6_4_confusion_matrix.png", bbox_inches="tight", facecolor="white")
plt.show()


## Summary · 학습 흐름 정리

`from ... import ...`로 클래스를 불러오고 `model = ...()`로 **학습 전 모델 객체를 생성**합니다. `fit()`으로 학습한 뒤 `predict()`로 입력의 결과를 계산하고, 실제 정답과 비교하여 평가합니다. 여러 후보를 비교할 때에는 Validation을 사용하고 Test는 마지막 평가에 남겨 둡니다.

**Exercise:** ZIP의 `practice/exercise_01_regression.py`, `exercise_02_classification.py`, `exercise_05_pipeline.py`를 참고하세요. 자세한 문제는 `README.md`의 Exercise에 있습니다.